# Paired GRU seed comparison

Compare reference and patience twelve with seeds 42, 43, and 44.
Keep the data split, features, model size, device, and epoch cap fixed.
The suite runs six fresh models. It uses Adam; no OPAL agent is involved.
Planning is the default. Enable RUN_TRAINING only when ready.
Reports skip training and read saved results.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("gru_suite_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/gru-seeds.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/gru-seed-comparison"  # Select the PR branch before merge
RUN_SETUP = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)


## Optional setup

Use the existing checkout locally. Set `SLEEP_CLASSIFIER_ROOT` before starting Jupyter if needed.
Enable setup for a fresh environment. Select the current PR branch before its merge.
Install uv through the Astral installer when uv is absent.
Prepare the cohort through notebook 07 before training this suite.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Inspect the plan

This command reads configuration files only. It does not load participant data or train models.
Automatic device selection prefers CUDA, then MPS, then CPU when training starts.
All six runs use that resolved device. This avoids comparing the earlier CPU run with a changed MPS configuration.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.gru_suite \
    --config configs/gru-seeds.yaml --plan
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Suite planning failed")


## Run the suite when ready

Set `RUN_TRAINING = True` and execute this cell once.
The suite runs six trials sequentially, with at most 120 epochs in total.
Early stopping can reduce this total. Each trial restores its best validation checkpoint.
The suite saves progress after each trial and refuses an existing output directory.
After a failure, inspect the partial summary. Choose a new output directory before another full run.

Repeated validation comparisons can overfit these three participants. Treat results as exploratory and confirm later with grouped evaluation.


In [ ]:
if RUN_TRAINING:
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.gru_suite \
        --config configs/gru-seeds.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Suite training failed; inspect the saved partial summary")


## Inspect saved comparisons

Read accuracy and macro-F1 together. Inspect each trial's class metrics before choosing a candidate.
Compare the paired differences across matching seeds.
Seed variation does not measure participant sensitivity.


In [ ]:
import pandas as pd
import yaml

suite_path = ROOT / "configs/gru-seeds.yaml"
suite_settings = yaml.safe_load(suite_path.read_text())
summary_path = (suite_path.parent / suite_settings["output_dir"] / "summary.json").resolve()
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    logger.info("Suite status: %s; device: %s", summary["status"], summary["device"])
    results = pd.DataFrame(summary["results"])
    if not results.empty:
        display(results[["name", "accuracy", "macro_f1", "best_epoch", "epochs_completed"]])
        for result in summary["results"]:
            logger.info("Class metrics: %s", result["name"])
            display(pd.DataFrame(result["classes"]).T)

    if "seed_comparison" in summary:
        display(pd.DataFrame(summary["seed_comparison"]["recipes"]).T)
        display(pd.DataFrame(summary["seed_comparison"]["paired"]).T)
else:
    logger.info("No suite results exist. Planning did not train a model.")


In [ ]:
if summary_path.exists() and summary["results"]:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    results.set_index("name")[["accuracy", "macro_f1"]].plot.bar(ax=axes[0], ylim=(0, 1))
    axes[0].set(xlabel="Trial", ylabel="Validation score")
    recall = pd.DataFrame({
        result["name"]: {stage: score["recall"] for stage, score in result["classes"].items()}
        for result in summary["results"]
    }).T
    recall.columns = ["Wake", "Light", "Deep", "REM"]
    recall.plot.bar(ax=axes[1], ylim=(0, 1))
    axes[1].set(xlabel="Trial", ylabel="Class recall")
    figure.tight_layout()
    plt.show()

## Export the report

Export a fresh HTML report after training. The export does not rerun training.
Keep the source notebook output-free.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/09_gru_seed_comparison.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Report export failed")